# Qwen3-4B-Thinking-2507 -- Fine-tune on ViNumQA (Unsloth)

Fine-tune `unsloth/Qwen3-4B-Thinking-2507` on the ViNumQA (VLSP-2025 numerical reasoning) train split,
using teacher-distilled reasoning traces as the `<think>` target, followed by the gold FinQA-style
computation program.

**Data**: set `DATASET_VARIANT` in the data-prep cell to pick which dataset to train on. Two are
GOLD-MERGED (every one of the 2993 train samples kept; `reasoning_trace` is null for samples the
teacher did not verifiably solve, trained as bare-program examples with an empty think block, so no
training data is lost); two are the RAW teacher output, NOT merged -- only the verified samples are in
the file at all, so every training example carries a real reasoning trace and the sample count is
smaller:

| `DATASET_VARIANT` | trace language | verification filter | merged with full split? | train n | valid n |
|---|---|---|---|---|---|
| `v6` | Vietnamese | union (strict OR PA) | yes (2364/2993, 79.0% have a trace) | 2993 | 584 |
| `v6_en` | English | union (strict OR PA) | yes (2187/2993, 73.1% have a trace) | 2993 | 584 |
| `pa` | Vietnamese | PA-scorer only (strictest) | **no** -- 100% of rows have a trace | 2290 | 455 |
| `pa_en` | English | PA-scorer only (strictest) | **no** -- 100% of rows have a trace | 2123 | 450 |

All four come from the same independent-solve teacher run
(`distill-reasoning-trace/gemma-4-31b-conr-trace-gen-independent-solve*.ipynb`, teacher: gemma-4-31B-it),
where the teacher got only context+question and had to derive the program itself, never the gold
program or answer -- "verified" means the teacher's own derived program either matched gold exactly
(after number-format normalization, e.g. `100` == `100.00`, the "strict" test) or matched under the
official symbolic PA scorer (algebraically equivalent, e.g. commuted arguments). The EN variants carry
an English-language reasoning narration while context/question/program stay Vietnamese (quoted table
labels and source-text evidence inside the English trace are also kept in Vietnamese, per the teacher
prompt's rule). `v6`/`v6_en` use the union filter (recommended default) and keep every sample via the null-trace
merge; `pa`/`pa_en` use the strictest test alone and drop unverified samples entirely instead of
keeping them as bare-program rows -- a lower-n but potentially higher-precision alternative.

Unlike an earlier dataset version, **no answer-conditioned rationalization pass is included** here:
traces where the teacher was given the gold answer as a disambiguation hint (to resolve dataset
labelling ambiguities the question's wording alone cannot settle) were deliberately excluded after
manual review found roughly half of them papered over inconsistent gold labelling conventions rather
than fixing genuine teacher reasoning errors -- every trace in every variant reflects the teacher
solving completely unaided.

Same training setup as `qwen3-4b-stf-w-reasoning-trace.ipynb` in this folder -- the base checkpoint
and the trace source differ. `Qwen3-4B-Thinking-2507` is a thinking-only refresh of Qwen3-4B (always emits a
`<think>` block, cannot be disabled; per Qwen's own release notes it has "increased thinking length" vs
the original hybrid Qwen3-4B). Since it natively tends to think longer, `max_new_tokens` at inference/eval
time is increased below to reduce the risk of truncating a longer generated trace -- everything else
(prompt format, LoRA config, training hyperparameters, `MAX_SEQ_LENGTH`) is unchanged, since those are
governed by the training *data*, which is identical in shape across variants.
Run on Kaggle (T4/P100 free tier). For a single continuous session on Modal instead (no 12h cap, so
train+eval can stay in one notebook), see `qwen3-4b-thinking-2507-stf-w-reasoning-trace-modal.ipynb`.

**This notebook only trains and saves the adapter.** Evaluation lives in
`qwen3-4b-thinking-2507-eval-only.ipynb`, which loads the adapter this notebook produces.
They are split because one measured run spent ~11.2h on training alone against Kaggle's 12h
session limit, so the eval loop was cut off partway and the entire run -- adapter included --
was lost. Add this notebook's output as a data source to the eval notebook.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
from huggingface_hub import login

# Optional: only needed if you plan to push the adapter/merged model to the Hub.
# On Kaggle: Add-ons > Secrets > add "HF_TOKEN", then attach it to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("No HF_TOKEN found -- skipping login (fine unless you want to push_to_hub).")

### Load base model + LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

# Sized from the v5 data: the longest fully-formatted training sample is
# ~4.7k tokens (system + context + question + trace + program), so 5678 was
# enough for training. Inference needs the same window to ALSO fit the
# generated answer on top of the prompt, and a 4.7k prompt + 2048 new tokens
# overflows 5678 -- hence 8192. Raising this is cheap: batches are padded to
# their own longest sequence, not to MAX_SEQ_LENGTH.
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data prep

Load the official 3-way ViNumQA split and reuse the exact same context formatting (pre_text / markdown
table / post_text) and system prompt as the 0-shot/1-shot/SFT notebooks, so results stay comparable.

In [ ]:
import glob
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Pick which dataset variant to train on -- everything below (paths, OUTPUT_DIR,
# ADAPTER_DIR) derives from this one switch, so running a different variant
# never needs any other edit in this notebook.
#   "v6"    -- VI, union filter (strict OR PA), GOLD-MERGED: every one of the 2993
#              train samples is kept, reasoning_trace is null for the 629 the
#              teacher did not verifiably solve (trained as bare-program). 2364/2993 (79.0%) verified.
#   "v6_en" -- same merge, EN reasoning trace (VI context/program unchanged), union filter. 2187/2993 (73.1%).
#   "pa"    -- VI, PA-scorer-only filter (strictest), NOT merged: only the 2290
#              samples the teacher verifiably solved are in the file at all --
#              no null/bare-program rows, unlike v6/v6_en.
#   "pa_en" -- same as "pa", EN reasoning trace. Only the 2123 verified samples, not merged.
DATASET_VARIANT = "v6"

_VARIANT_FILES = {
    "v6":    ("train_mixed_reasoning_v6.json", "valid_mixed_reasoning_v6.json"),
    "v6_en": ("train_mixed_reasoning_v6_en.json", "valid_mixed_reasoning_v6_en.json"),
    "pa":    ("train_with_reasoning_trace_pa.json", "valid_with_reasoning_trace_pa.json"),
    "pa_en": ("train_with_reasoning_trace_pa_en.json", "valid_with_reasoning_trace_pa_en.json"),
}
TRAIN_FILE, VALID_FILE = _VARIANT_FILES[DATASET_VARIANT]

# v6*/pa* files live in a NEW Kaggle dataset (built from the gemma-4-31B-it teacher,
# not the teammate's v5 one) -- update _CANDIDATES to wherever you attach it, or
# leave the glob fallback to find it automatically among attached datasets. The two
# distill-reasoning-trace output dirs are only relevant to "pa"/"pa_en" (raw teacher
# output, un-merged); "v6"/"v6_en" live in datasets/ViNumQA instead -- harmless to
# list all of them, since TRAIN_FILE's own name disambiguates which one has it.
_CANDIDATES = [
    Path("/kaggle/input/vlsp2025-vinumqa-v6"),  # <- rename to match your uploaded dataset's slug
    Path("datasets/ViNumQA"),                    # local repo path, if running outside Kaggle
    Path("notebooks/vinumqa/distill-reasoning-trace/outputs/conr_trace_gemma_independent_solve"),
    Path("notebooks/vinumqa/distill-reasoning-trace/outputs/conr_trace_gemma_independent_solve_en"),
]
DATA_DIR = next((p for p in _CANDIDATES if (p / TRAIN_FILE).exists()), None)
if DATA_DIR is None:
    _hits = glob.glob(f"/kaggle/input/**/{TRAIN_FILE}", recursive=True)
    if _hits:
        DATA_DIR = Path(_hits[0]).parent
if DATA_DIR is None:
    raise FileNotFoundError(
        f"{TRAIN_FILE} not found. Upload {{train,valid}} data for DATASET_VARIANT={DATASET_VARIANT!r} "
        "(and test.json) as a Kaggle dataset and attach it, or update _CANDIDATES above."
    )
print(f"Using data from: {DATA_DIR}  (variant={DATASET_VARIANT})")

# test.json is resolved independently of DATA_DIR: for "pa"/"pa_en" the train/valid
# files live in the raw teacher-output dir, which does NOT contain test.json (that
# only lives in datasets/ViNumQA). Attaching both as Kaggle datasets sidesteps this
# in practice, but resolving it separately keeps the notebook correct either way.
TEST_DIR = next((p for p in _CANDIDATES if (p / "test.json").exists()), None)
if TEST_DIR is None:
    _hits = glob.glob("/kaggle/input/**/test.json", recursive=True)
    if _hits:
        TEST_DIR = Path(_hits[0]).parent
if TEST_DIR is None:
    raise FileNotFoundError("test.json not found -- attach a Kaggle dataset containing it, or update _CANDIDATES above.")

train_df = pd.read_json(DATA_DIR / TRAIN_FILE)
valid_df = pd.read_json(DATA_DIR / VALID_FILE)
test_df = pd.read_json(TEST_DIR / "test.json")
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

# qa.reasoning_trace is null for samples the teacher did not solve correctly;
# those are trained as bare-program examples (see build_conversation below).
n_trace = sum(1 for r in train_df["qa"] if r.get("reasoning_trace"))
print(f"train samples carrying a verified reasoning trace: {n_trace} / {len(train_df)} "
      f"({100 * n_trace / len(train_df):.1f}%)")

In [ ]:
train_df

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_reasoning_trace(sample):
    # None for samples without a verified trace -- kept as None (not "")
    # so build_conversation can tell the two cases apart.
    return sample["qa"].get("reasoning_trace")

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_reasoning_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["reasoning_trace_processed"] = df.apply(processing_reasoning_trace, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question", "reasoning_trace_processed",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "post_text", "question", "reasoning_trace", "program", "answer"]
    return df

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "post_text", "question", "program", "answer"]
    return df

train_df = process_reasoning_split(train_df)
valid_df = process_reasoning_split(valid_df)

test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""


### Build the conversational dataset

In [ ]:
# Qwen3-4B-Thinking-2507 is thinking-only: its chat template always opens a
# <think> block and `enable_thinking=False` is not supported (per the model
# card). So the two training modes are built by hand in the assistant content
# rather than via the template flag:
#   - verified trace  -> "{trace}\n</think>\n\n{program}"
#   - no trace (bare) -> "\n</think>\n\n{program}"  (empty think block)
# Neither writes a literal opening <think>; the template emits it, which is
# also why this model's own generations contain only a closing </think>.
def build_conversation(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )

    program = str(row["program"]).strip()
    trace = row["reasoning_trace"]
    # Treat this as a trace ONLY if it is a genuine non-empty string. Missing
    # values arrive as None, float('nan') or pd.NA depending on the pandas
    # version and column dtype, and str() on the latter two yields 'nan' /
    # '<NA>' -- non-empty strings that would silently be trained as the
    # reasoning text. The isinstance check rejects all of them regardless of
    # pandas version.
    if isinstance(trace, str) and trace.strip():
        assistant_content = f"{trace.strip()}\n</think>\n\n{program}"
    else:
        assistant_content = f"\n</think>\n\n{program}"

    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_content},
    ]


def make_text_dataset(df):
    conversations = [build_conversation(row) for _, row in df.iterrows()]
    texts = tokenizer.apply_chat_template(conversations, tokenize=False)
    return texts


train_texts = make_text_dataset(train_df)
valid_texts = make_text_dataset(valid_df)
print(len(train_texts), len(valid_texts))

# Sanity-check the two modes before training: each formatted sample must
# contain exactly one <think> and one </think>. If the counts are off, the
# template is inserting tags differently than assumed above and the assistant
# content needs adjusting -- do not train on it as-is.
_bad = [t for t in train_texts[:200] if t.count("<think>") != 1 or t.count("</think>") != 1]
print(f"malformed think tags in first 200 samples: {len(_bad)} (must be 0)")

_recs = train_df.to_dict("records")
_has = lambda r: isinstance(r["reasoning_trace"], str) and r["reasoning_trace"].strip()
_with = next((i for i, r in enumerate(_recs) if _has(r)), None)
_without = next((i for i, r in enumerate(_recs) if not _has(r)), None)
# "pa"/"pa_en" are un-merged (every row has a real trace), so there is no bare
# example to contrast against -- print whichever of the two exist instead of
# assuming both do (that assumption held for v6/v6_en, not for pa/pa_en).
if _with is not None:
    print("\n===== WITH trace (tail) =====")
    print(train_texts[_with][-900:])
if _without is not None:
    print("\n===== BARE / empty think (tail) =====")
    print(train_texts[_without][-400:])
else:
    print("\n(no bare/empty-think example in this DATASET_VARIANT -- every row has a verified trace)")

In [ ]:
# Compute the max token length across train/valid samples after templating,
# so max_seq_length in SFTConfig can be set to actually cover the longest
# reasoning-trace samples instead of guessing (or silently truncating them).
def compute_token_lengths(texts):
    return [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in texts]

train_lengths = compute_token_lengths(train_texts)
valid_lengths = compute_token_lengths(valid_texts)

import numpy as np

for name, lengths in [("train", train_lengths), ("valid", valid_lengths)]:
    lengths = np.array(lengths)
    print(f"{name}: n={len(lengths)}  max={lengths.max()}  p99={np.percentile(lengths, 99):.0f}  "
          f"p95={np.percentile(lengths, 95):.0f}  mean={lengths.mean():.0f}")

overall_max = max(max(train_lengths), max(valid_lengths))
print(f"\nOverall max token length: {overall_max}")

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_texts}).shuffle(seed=3407)
valid_dataset = Dataset.from_dict({"text": valid_texts})
train_dataset, valid_dataset

<a name="Train"></a>
### Train the model

We mask the loss to only the assistant turn (`train_on_responses_only`) so the model isn't penalized for
"predicting" the system prompt / context / question -- standard practice, and important here since the
context block can be much longer than the program string we actually want it to learn to produce.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = f"/kaggle/working/qwen3-4b-thinking-2507-vinumqa-sft-{DATASET_VARIANT}"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 8,   # effective batch size = 16
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 20,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Train the model. To resume a run, set `trainer.train(resume_from_checkpoint = True)`.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference

In [ ]:
FastLanguageModel.for_inference(model)

_row = test_df.iloc[0]
_prompt = USER_MESSAGE_FRAME.format(
    pre_text=_row["pre_text"], table=_row["table"], post_text=_row["post_text"], question=_row["question"],
)
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": _prompt},
]
text = tokenizer.apply_chat_template(
    # No enable_thinking flag: Thinking-2507 is thinking-only and the model
    # card states the flag is no longer required. Omitting it here keeps the
    # inference formatting byte-identical to how training data was templated.
    messages, tokenize=False, add_generation_prompt=True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    # The trained answer (trace + program) tops out near 1k tokens in the v5
    # data, so 2048 is ample; 8192 would have exceeded the context window
    # once added to a long prompt.
    max_new_tokens=2048,
    temperature=0.6, top_p=0.95, top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)
print("\nGold program:", _row["program"], "| Gold answer:", _row["answer"])

<a name="Save"></a>
### Saving

Saves LoRA adapters locally. Uncomment the `push_to_hub` lines (and set `HF_USERNAME`) to also upload.
See the merged-16bit / GGUF cells below for other export options (same as the reference notebook).

In [ ]:
ADAPTER_DIR = f"/kaggle/working/qwen3-4b-thinking-2507-vinumqa-sft-adapter-{DATASET_VARIANT}"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter to {ADAPTER_DIR}")

# Also push to the Hugging Face Hub, if a token is available.
#
# /kaggle/working is only published when the whole Save Version run finishes.
# A run that hits the 12h session limit is marked failed and its output is
# normally discarded -- which is how an 11h training run can be lost in full.
# Pushing here, immediately after training, puts the adapter somewhere that
# survives a later timeout, and the eval notebook can load it straight from the
# Hub instead of waiting on this notebook's output.
#
# Set HF_USERNAME and add HF_TOKEN under Add-ons > Secrets to enable this.
HF_USERNAME = None  # e.g. "your-hf-username"

if HF_TOKEN and HF_USERNAME:
    repo = f"{HF_USERNAME}/qwen3-4b-thinking-2507-vinumqa-sft-adapter-{DATASET_VARIANT}"
    model.push_to_hub(repo, token=HF_TOKEN)
    tokenizer.push_to_hub(repo, token=HF_TOKEN)
    print(f"Pushed adapter to https://huggingface.co/{repo}")
else:
    print("Skipping Hub push (needs HF_USERNAME above + HF_TOKEN secret). "
          "The adapter then only survives if this run finishes without timing out.")

In [ ]:
# Merge to 16bit / 4bit, or export GGUF for llama.cpp -- disabled by default.
if False:
    model.save_pretrained_merged("qwen3-4b-thinking-2507-vinumqa-sft-16bit", tokenizer, save_method="merged_16bit")
if False:
    model.save_pretrained_merged("qwen3-4b-thinking-2507-vinumqa-sft-4bit", tokenizer, save_method="merged_4bit")
if False:
    model.save_pretrained_gguf("qwen3-4b-thinking-2507-vinumqa-sft", tokenizer, quantization_method="q4_k_m")

## Done

The LoRA adapter is saved under `/kaggle/working`. Commit this notebook (Save Version) so the
output is retained, then attach it as a data source in
`qwen3-4b-thinking-2507-eval-only.ipynb` to score PA/EA.